# Interpreting an audit report

A guided tour of the results tree and the report sections, using the bundled Grid Stability audit.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd() if (Path.cwd() / 'datasets').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from synthaudit import Audit
df = pd.read_csv(ROOT / 'datasets' / 'grid_stability.csv')
results = Audit(df, target='stabf', name='grid').run(verbose=False)
list(results.keys())

['profile',
 'identity',
 'determinism',
 'causal',
 'leakage',
 'taxonomy',
 'scoring',
 'recommendations',
 'meta']

**Read in this order.** 1) `scoring`: the verdict and which pillar drove it. 2) `leakage.findings`: the prioritized evidence. 3) `identity`: the recovered relations with dispositions. 4) `taxonomy`: what role every column plays. 5) `recommendations`: what to actually do.

In [2]:
s = results['scoring']
print('grade', s['grade'], '| BTI', s['bti'])
for k, v in s['pillars'].items():
    print(f'  {k}: {v}')
s['evidence']['label_evidence']

grade F | BTI 0.2312
  L: 0.0
  F: 0.8462
  H: 1.0
  R: 0.7179
  I: 1.0


["stabf == 'unstable'  iff  stab > -7.88345e-07"]

In [3]:
for f in results['leakage']['findings']:
    print(f"[{f['severity']:8s}] {f['kind']:26s} {f['detail'][:70]}")

[critical] derived_target             Target 'stabf' is (near-)recoverable by threshold_derivation: stabf ==
[critical] label_components           Columns ['stab'] functionally participate in the target's generating r
[critical] single_feature_dominance   Feature 'stab' alone achieves 1.0000 (AUC/acc) on 'stabf' — shortcut o


In [4]:
roles = pd.DataFrame([
    {'column': c, **r} for c, r in results['taxonomy']['roles'].items()
])
roles.groupby('role').size()

role
derived_deterministic     1
input                    11
label_component           1
target                    1
dtype: int64

In [5]:
rec = results['recommendations']
print('drop      :', rec['drop_columns'])
print('quarantine:', rec['quarantine_columns'])
print('warnings  :', rec['protocol_warnings'])
print('tasks     :', rec['suggested_tasks'])

drop      : ['p4', 'stab']
quarantine: []
warnings  : ["Target 'stabf' is (near-)derived from shipped columns. Any reported model score on it is a claim about equation recovery, not about learning."]
tasks     : ["Predict 'p1' (honest out-of-sample r2≈0.65).", "Predict 'g3' (honest out-of-sample r2≈0.51).", "Predict 'g2' (honest out-of-sample r2≈0.51)."]


**Severity semantics.** `critical` means the posed task is compromised (derived target, label components, contamination). `high` means the protocol is compromised (schedule leakage, heavy duplication). `medium/low` are hygiene. **Dispositions** separate leakage from legitimate structure; a physics constraint among inputs is reported, not punished. When you file an issue or a finding, paste the `equation` and its verification, not just the grade.